In [ ]:

#      packages       
# ======================================
# 

import os
os.environ["KERAS_BACKEND"] = "tensorflow"


import numpy                    as np
import pandas                   as pd
import matplotlib.pyplot        as plt
import seaborn                  as sns

from matplotlib.ticker          import StrMethodFormatter

import re
import time

from IPython.display            import display, HTML, Markdown


from sklearn.model_selection    import train_test_split
from sklearn.preprocessing      import LabelEncoder
from sklearn.metrics            import classification_report, confusion_matrix, f1_score

import keras
import keras_hub                

from keras                      import mixed_precision                   
mixed_precision.set_global_policy("mixed_float16")



# --- Hugging Face Datasets
from datasets import load_dataset, DatasetDict
from datasets.features import ClassLabel


##_____ just some helper _____ ##: 

def view_content(df):
    if isinstance(df, pd.Series):
        df = df.to_frame()

    with pd.option_context("display.max_colwidth", None):
        html = df.to_html(
            max_rows=None,
            max_cols=None,
            escape=True,
        )

    display(HTML(
        '<div style="width:100%; overflow-x:auto;">'
        '<style>'
        '.full-text-table td, .full-text-table th {'
        '    white-space:pre-wrap;'
        '    overflow-wrap:anywhere;'
        '    min-width:100px;'
        '    max-width:500px;'
        '    text-align:left;'
        '    vertical-align:top;'
        '}'
        '</style>'
        + html.replace('class="dataframe"', 'class="dataframe full-text-table"')
        + '</div>'
    ))

def format_hms(seconds):
    """Format elapsed seconds as H:MM:SS."""
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

                
#      dataset       
# ======================================
# 

URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
huff_all = load_dataset("json", data_files=URL, split="train")

df = huff_all.to_pandas()



In [ ]:

                
#      additional function - pulling authors' specialties       
# ======================================
####____◄ many authors have specialties that they most likely write one category only.

possible_markers_for_specialty =    ['contributor', 'author', 'specialist']
extra_words_to_replace =            ['editor', 'author']



def _build_pattern(words: list, whole_word: bool = False) -> str:
    joined = "|".join(re.escape(w) for w in words if w)
    if not joined:
        return ""
    return rf"\b(?:{joined})\b" if whole_word else joined


def pull_author_specialty(
    author,
    marker: list = possible_markers_for_specialty,
    extra_words: list = extra_words_to_replace,
) -> str:
    ## __Return "" for missing or non-text input
    if not isinstance(author, str):
        return ""

    ## __Return "" when no marker pattern exists
    pattern_marker = _build_pattern(marker)
    if not pattern_marker:
        return ""

    ## __Match the first marker, ignoring capitalization
    match = re.search(pattern_marker, author, flags=re.IGNORECASE)
    if match is None:
        return ""

    ## __Extract text after the marker and convert literal \n to a space
    specialty = author[match.end():].replace("\\n", " ")

    ## __Remove unwanted words, if any were provided
    pattern_after = _build_pattern(extra_words, whole_word=True)
    if pattern_after:
        specialty = re.sub(pattern_after, "", specialty, flags=re.IGNORECASE)

    ## __Normalize spaces and trim punctuation
    specialty = " ".join(specialty.split()).strip(",:;- ")

    return specialty



                
#       make function for creating the feature / input text      
# ======================================
# 

def combining_news_text(df)->pd.DataFrame:
    return (
        df 
        .assign(
            cleaned_headline = lambda x: (
                x["headline"] 
                .fillna("")                     ####____◄ make any NA to be empty... not that needed
                .astype("string")               ####____◄ 
                .str.replace(r"\s+", " ", regex = True)     ####____◄ just fixing escape code
                .str.strip()
            ),

            cleaned_content = lambda x: (
                x["short_description"]
                .fillna("") 
                .astype("string")
                .str.replace(r"\s+", " ", regex = True)
                .str.strip()
            ),

            author_spec = lambda x: (
                x['authors']
                .apply(pull_author_specialty)
            )
        )

        .assign(
            combo_text_H_C_     = lambda x: (  
                            x["cleaned_headline"] + "\n" + x["cleaned_content"]
                            ).str.strip()
                ,
            combo_text_H_C_A_   = lambda x: (                       ####____◄ including author's specialty
                            x["cleaned_headline"] + "\n" + x["cleaned_content"] + "\n" + x["author_spec"]
                            ).str.strip()
        )
    )

In [ ]:
#### ----------------------------------------------------- ####
                
#      PREPROCESSING       
# ======================================
# 

##_____ CREATE COMBO TEXT _____ ##: using the functions above

df_cleaned_1 = (
    df 
    .pipe(combining_news_text)          ####____◄ create the 2 combo text columns
    .query("combo_text_H_C_ != '' ")    ####____◄ cleaning up rows without any text/input
    .dropna(subset = ['category'])
)


##_____ CLEANUP CONFLICTING/DUPS _____ ##: removing those with same text but diff labels

conflicting_texts = (
    df_cleaned_1
    .groupby("combo_text_H_C_")["category"]
    .nunique()
    .loc[lambda x: x > 1]
    .index
)


df_cleaned_2 = (
    df_cleaned_1
    .loc[lambda x: ~x["combo_text_H_C_"].isin(conflicting_texts)]
    .drop_duplicates(subset=["combo_text_H_C_"], keep="last") ####____◄ for now, selecting that the last repeat is the correct label
    .reset_index(drop=True)
)



##### •••••• ####
##### REVIEWING THE PREPROCESSING 
df_conflicts = (
    df_cleaned_1
    .loc[lambda x: x["combo_text_H_C_"].isin(conflicting_texts)]
    .sort_values(["combo_text_H_C_", "category"])
)

# display(df_conflicts[["headline", "combo_text_H_C_", "category", "date"]].iloc[10:30])


# print(f"Dataset size after removing empyt text, but not duplicates: {len(df_cleaned_1)}")
# print(f"Dataset size after remoivng duplicates: {len(df_cleaned_2)}")

In [ ]:
            
#      PREPROCESSING       2
# ======================================
# 

##_____ CREATE FINAL LABELS _____ ##: merging some of the overlaps or low counts
label_groups = {
    'ARTS & CULTURE'        :   [ 'ARTS', 'ARTS & CULTURE', 'CULTURE & ARTS'],
    'WORLD NEWS'            :   [ 'WORLDPOST','WORLD NEWS' ,'THE WORLDPOST'   ],
    'PARENTING'             :   [ 'PARENTS', 'PARENTING'   ],
    'STYLE & BEAUTY'        :   [ 'STYLE', 'STYLE & BEAUTY'   ],
    'FOOD & DRINK'          :   [ 'TASTE'  , 'FOOD & DRINK',   ],
    'ENVIRONMENT'           :   [ 'GREEN', 'ENVIRONMENT'   ],
    'WELLNESS'              :   [ 'HEALTHY LIVING' , 'WELLNESS'   ],  
}

## __Convert grouped labels into a lookup dictionary
label_mapping = {
    original: merged
    for merged, originals in label_groups.items()
    for original in originals
}

df_cleaned_3 = (
    df_cleaned_2
    .assign(
        cleaned_news_category = lambda x: (
            x["category"]
            .map(label_mapping)
            .fillna(x["category"])
        )
    )
)


##### •••••• ####
##### REVIEWING THE PREPROCESSING 

# view_content(df_cleaned_3.sample(3)[["category", "cleaned_news_category", "combo_text_H_C_A_"]])

In [ ]:
#### ----------------------------------------------------- ####

                
#      SPLITTING       
# ======================================
# 



SEED = 42
rng = np.random.default_rng(SEED)
keras.utils.set_random_seed(SEED)


TEXT_INPUT_COL = "combo_text_H_C_A_"
NEWS_CATEGORY = "cleaned_news_category"

df_model = df_cleaned_3.copy()



##_____ splitting to 80-20 first _____ ##: 
df_train, df_holdout = (
    train_test_split(
        df_model,
        test_size       = 0.20 , 
        stratify        = df_model[NEWS_CATEGORY],
        random_state    = SEED
    )
)

##_____ splitting again, the holdout, for validation _____ ##: 
df_val, df_test = (
    train_test_split(
        df_holdout,
        test_size       = 0.50 ,            ####____◄ so.. 20% into 10% and 10%
        stratify        = df_holdout[NEWS_CATEGORY],
        random_state    = SEED
    )
)


##_____ label & prep sets _____ ##: 

label_encoder = LabelEncoder()  ####____◄ category-to-integer mapping

y_train     = label_encoder.fit_transform(df_train[NEWS_CATEGORY])
y_val       = label_encoder.transform(df_val[NEWS_CATEGORY])
y_test      = label_encoder.transform(df_test[NEWS_CATEGORY])

x_train     = df_train[TEXT_INPUT_COL].to_numpy(dtype = str)
x_val       = df_val[TEXT_INPUT_COL].to_numpy(dtype = str) 
x_test      = df_test[TEXT_INPUT_COL].to_numpy(dtype = str)





---

# model

In [ ]:

MAX_LEN = 128
BATCH = 32
EVAL_BATCH = 64
EPOCHS = 3
LR = 2e-5

NUM_NEWS_CATEGORY = len(label_encoder.classes_)

                
#      PREP FUNCTIONS       
# ======================================
# 

def make_model(
        preset      = "distil_bert_base_en_uncased" ,
        max_len     = MAX_LEN, 
        lr          = LR
):
    preproc  = keras_hub.models.DistilBertTextClassifierPreprocessor.from_preset(
        preset, 
        sequence_length = max_len
    )

    model  = keras_hub.models.DistilBertTextClassifier.from_preset(
        preset, 
        num_classes = NUM_NEWS_CATEGORY ,
        preprocessor  = preproc
    )

    model.compile(
        optimizer   = keras.optimizers.Adam(lr) , 
        loss        = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
        metrics     = [keras.metrics.SparseCategoricalAccuracy(name = "acc")]
    )

    return model


##### •••••• ####

def simple_train(title, **build_kwargs):
    model = make_model(**build_kwargs)

    time0 = time.perf_counter()

    hist = model.fit(
        x_train, y_train,
        validation_data     = (x_val, y_val),
        epochs              = EPOCHS , 
        batch_size          = BATCH,
        callbacks           = [keras.callbacks.EarlyStopping(
                                    monitor = "val_loss",
                                    patience = 2, 
                                    restore_best_weights = True
                                    )],
        verbose             = 1
    )

    history_df = (pd.DataFrame(hist.history)
                  .assign(epoch = lambda x: np.arange(1, len(x) + 1))
    )

    best = history_df['val_loss'].idxmin()


    pred = model.predict(
                    x_val,
                    batch_size  = EVAL_BATCH,
                    verbose     = 0
                    ).argmax(axis = -1)

    row = {
        "title"     : title, 
        "best_epoch": int(history_df.loc[best, "epoch"]),

        "val_loss": history_df.loc[best, "val_loss"], 
        "val_acc": history_df.loc[best, "val_acc"],
        "val_macro_f1": f1_score(y_val, pred, average = "macro"),
        "seconds": format_hms(time.perf_counter() - time0)
        
    }

    return model, history_df, row



In [ ]:
model, hist_df, row = simple_train("distilbert_len128")

results_df = pd.DataFrame([row]).set_index("title")
